In [17]:
# load packages
import numpy as np
import pandas as pd
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier,
    ExtraTreesClassifier,
)
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
import networkx as nx
from lightgbm import LGBMClassifier, early_stopping
from sklearn.model_selection import StratifiedKFold
import torch
from torch import nn
from torch.nn import functional as F

In [2]:
# read in data
sample_submission = pd.read_csv("data/sample_submission.csv") # 15329 rows × 2 columns
train = pd.read_csv("data/train.csv") # 141772 rows × 168 columns
test = pd.read_csv("data/test.csv") # 15329 rows × 168 columns
edges = pd.read_csv("data/txs_edgelist.csv") # 234355 rows × 2 columns

# Data Exploration

In [3]:
print("Train:", train.shape)
print("Test:", test.shape)
print("Edges:", edges.shape)
print("\n")
print("Train columns:", train.columns.tolist())
print("\n")
print("Test columns:", test.columns.tolist())
print("\n")
print("edges columns:", edges.columns.tolist())
print("\n")
print("train labels:", train["label"].value_counts(dropna=False))
print("\n")
print("Train time range:", train["time_step"].min(), train["time_step"].max())
print("Test time range:", test["time_step"].min(), test["time_step"].max())

Train: (141772, 168)
Test: (15329, 168)
Edges: (234355, 2)


Train columns: ['txId', 'time_step', 'feat_1', 'feat_2', 'feat_3', 'feat_4', 'feat_5', 'feat_6', 'feat_7', 'feat_8', 'feat_9', 'feat_10', 'feat_11', 'feat_12', 'feat_13', 'feat_14', 'feat_15', 'feat_16', 'feat_17', 'feat_18', 'feat_19', 'feat_20', 'feat_21', 'feat_22', 'feat_23', 'feat_24', 'feat_25', 'feat_26', 'feat_27', 'feat_28', 'feat_29', 'feat_30', 'feat_31', 'feat_32', 'feat_33', 'feat_34', 'feat_35', 'feat_36', 'feat_37', 'feat_38', 'feat_39', 'feat_40', 'feat_41', 'feat_42', 'feat_43', 'feat_44', 'feat_45', 'feat_46', 'feat_47', 'feat_48', 'feat_49', 'feat_50', 'feat_51', 'feat_52', 'feat_53', 'feat_54', 'feat_55', 'feat_56', 'feat_57', 'feat_58', 'feat_59', 'feat_60', 'feat_61', 'feat_62', 'feat_63', 'feat_64', 'feat_65', 'feat_66', 'feat_67', 'feat_68', 'feat_69', 'feat_70', 'feat_71', 'feat_72', 'feat_73', 'feat_74', 'feat_75', 'feat_76', 'feat_77', 'feat_78', 'feat_79', 'feat_80', 'feat_81', 'feat_82', 'feat_83'

In [4]:
# step 2: check table consistency and feature quality

FEATURE_COLS = [f"feat_{i}" for i in range(1, 166)]
                
train_feature_cols = [c for c in train.columns if c.startswith("feat_")]
test_feature_cols = [c for c in test.columns if c.startswith("feat_")]

print("Train and test feature names match:",
      train_feature_cols == test_feature_cols)
print("Expected feature columns missing from train:",
      sorted(set(FEATURE_COLS) - set(train.columns)))
print("Expected feature columns missing from test:",
      sorted(set(FEATURE_COLS) - set(test.columns)))

common_features = [
    c for c in FEATURE_COLS
    if c in train.columns and c in test.columns
]

common_columns = [c for c in train.columns if c in test.columns]
dtype_mismatches = {
    c: (str(train[c].dtype), str(test[c].dtype))
    for c in common_columns
    if train[c].dtype != test[c].dtype
}
print("Train/test dtype mismatches:", dtype_mismatches)

print("Duplicate train txIds:", train["txId"].duplicated().sum())
print("Duplicate test txIds:", test["txId"].duplicated().sum())
print("Train/test txId overlap:", train["txId"].isin(test["txId"]).sum())

print("Edge rows with a missing endpoint ID:",
      edges[["txId1", "txId2"]].isna().any(axis=1).sum())

label_numeric = pd.to_numeric(train["label"], errors="coerce")
invalid_nonmissing_labels = train["label"].notna() & label_numeric.isna()

print("Non-numeric, non-missing labels:",
      invalid_nonmissing_labels.sum())
print("Observed numeric labels:",
      sorted(label_numeric.dropna().unique()))

numeric_features = [
    c for c in common_features
    if pd.api.types.is_numeric_dtype(train[c])
    and pd.api.types.is_numeric_dtype(test[c])
]
nonnumeric_features = sorted(set(common_features) - set(numeric_features))
print("Non-numeric feature columns:", nonnumeric_features)

if numeric_features:
    train_values = train[numeric_features].astype(float)
    test_values = test[numeric_features].astype(float)

    quality = pd.DataFrame(index=numeric_features)
    quality["train_missing"] = train_values.isna().sum()
    quality["test_missing"] = test_values.isna().sum()
    quality["train_infinite"] = np.isinf(train_values.to_numpy()).sum(axis=0)
    quality["test_infinite"] = np.isinf(test_values.to_numpy()).sum(axis=0)
    quality["train_unique_values"] = train_values.nunique(dropna=True)

    print("Total missing train feature values:",
          int(quality["train_missing"].sum()))
    print("Total missing test feature values:",
          int(quality["test_missing"].sum()))
    print("Total infinite train feature values:",
          int(quality["train_infinite"].sum()))
    print("Total infinite test feature values:",
          int(quality["test_infinite"].sum()))
    print("Constant features in train:",
          quality.index[quality["train_unique_values"] <= 1].tolist())

    display(
        quality.loc[
            (quality["train_missing"] > 0)
            | (quality["test_missing"] > 0)
            | (quality["train_infinite"] > 0)
            | (quality["test_infinite"] > 0)
        ]
    )

Train and test feature names match: True
Expected feature columns missing from train: []
Expected feature columns missing from test: []
Train/test dtype mismatches: {}
Duplicate train txIds: 0
Duplicate test txIds: 0
Train/test txId overlap: 0
Edge rows with a missing endpoint ID: 0
Non-numeric, non-missing labels: 0
Observed numeric labels: [np.float64(0.0), np.float64(1.0)]
Non-numeric feature columns: []
Total missing train feature values: 0
Total missing test feature values: 0
Total infinite train feature values: 0
Total infinite test feature values: 0
Constant features in train: []


,train_missing,test_missing,train_infinite,test_infinite,train_unique_values


In [5]:
# Check how labels change over time
if invalid_nonmissing_labels.any():
    raise ValueError("Some non-missing labels could not be converted to numbers.")

labeled = train.loc[label_numeric.notna(), ["txId", "time_step"]].copy()
labeled["label"] = label_numeric.loc[labeled.index].astype(int)

label_counts_by_time = (
    labeled.groupby(["time_step", "label"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=[0, 1], fill_value=0)
)
label_counts_by_time.columns = ["licit_count", "illicit_count"]

time_summary = (
    train.groupby("time_step")
    .size()
    .rename("total_transactions")
    .to_frame()
    .join(label_counts_by_time, how="left")
    .fillna(0)
)

time_summary["labeled_count"] = (
    time_summary["licit_count"] + time_summary["illicit_count"]
)
time_summary["unknown_count"] = (
    time_summary["total_transactions"] - time_summary["labeled_count"]
)
time_summary["illicit_rate"] = (
    time_summary["illicit_count"]
    / time_summary["labeled_count"].replace(0, np.nan)
)

display(time_summary.astype({"licit_count": int, "illicit_count": int,
                             "labeled_count": int, "unknown_count": int}))

,total_transactions,licit_count,illicit_count,labeled_count,unknown_count,illicit_rate
time_step,,,,,,
1,7880,2130,17,2147,5733,0.007918
2,4544,1099,18,1117,3427,0.016115
3,6621,1268,11,1279,5342,0.008600
4,5693,1410,30,1440,4253,0.020833
5,6803,1874,8,1882,4921,0.004251
6,4328,480,5,485,3843,0.010309
7,6048,1101,102,1203,4845,0.084788
8,4457,1098,67,1165,3292,0.057511
9,4996,530,248,778,4218,0.318766


## exploring edges

In [6]:
# Returns an integer representing the number of unique IDs
txId1_unique_count = edges['txId1'].nunique()
print("number of unique IDs in txId1 =", txId1_unique_count)

txId2_unique_count = edges['txId2'].nunique()
print("number of unique IDs in txId2 =", txId2_unique_count)

number of unique IDs in txId1 = 166345
number of unique IDs in txId2 = 148447


In [7]:
set_tx1 = set(edges['txId1'])
set_tx2 = set(edges['txId2'])

# 1. Nodes in both (Intermediate relays: receive AND send funds)
common_values = set_tx1.intersection(set_tx2)

# 2. Nodes in txId1 that DO NOT appear in txId2 (Source-only: only send funds)
only_in_tx1 = set_tx1 - set_tx2

# 3. Nodes in txId2 that DO NOT appear in txId1 (Dead-end sinks: only receive funds)
only_in_tx2 = set_tx2 - set_tx1

# 4. Total unique nodes in the edge list
all_edge_nodes = set_tx1 | set_tx2

print(f"Appear in BOTH (relays):           {len(common_values):,}")  # 111,023
print(f"ONLY in txId1 (sources/origins):   {len(only_in_tx1):,}")    # 55,322
print(f"ONLY in txId2 (dead-end sinks):    {len(only_in_tx2):,}")    # 37,424
print(f"Total unique nodes in edges:       {len(all_edge_nodes):,}") # 203,769

Appear in BOTH (relays):           111,023
ONLY in txId1 (sources/origins):   55,322
ONLY in txId2 (dead-end sinks):    37,424
Total unique nodes in edges:       203,769


In [8]:
# step 3: Check how labels change over time
# Map each node in train to its structural role in the graph
conditions = [
    train['txId'].isin(common_values),
    train['txId'].isin(only_in_tx1),
    train['txId'].isin(only_in_tx2)
]
choices = ['Relay (Both)', 'Source Only (txId1)', 'Dead-End Sink (txId2)']

train['node_role'] = np.select(conditions, choices, default='Isolated (No Edges)')

# Compare fraud rate across node roles on labeled data
labeled_train = train[train['label'].notna()]
role_summary = labeled_train.groupby('node_role')['label'].agg(
    total_labeled='count',
    illicit_count='sum',
    illicit_rate='mean'
).sort_values(by='illicit_rate', ascending=False)

print(role_summary)

                       total_labeled  illicit_count  illicit_rate
node_role                                                        
Dead-End Sink (txId2)           8320         1118.0      0.134375
Relay (Both)                   13212         1593.0      0.120572
Source Only (txId1)             9703          933.0      0.096156


In [9]:
# step 4: Check edge coverage, edge timing, degree, and neighbor feature coverage
# Remove rows with missing endpoint IDs, then count unique directed transaction pairs.
valid_edges = edges.dropna(subset=["txId1", "txId2"]).copy()
unique_edges = valid_edges[["txId1", "txId2"]].drop_duplicates()

print("Raw edge rows:", len(edges))
print("Edge rows with missing endpoint IDs:", len(edges) - len(valid_edges))
print("Duplicate directed edge rows:", len(valid_edges) - len(unique_edges))
print("Unique directed edges:", len(unique_edges))
print("Self-loops:", (unique_edges["txId1"] == unique_edges["txId2"]).sum())

# Map transaction IDs to their observed time steps.
node_times = pd.concat(
    [
        train[["txId", "time_step"]],
        test[["txId", "time_step"]],
    ],
    ignore_index=True,
).drop_duplicates("txId")

time_by_id = node_times.set_index("txId")["time_step"]

edge_times = unique_edges.copy()
edge_times["source_time"] = edge_times["txId1"].map(time_by_id)
edge_times["destination_time"] = edge_times["txId2"].map(time_by_id)

source_time_missing = edge_times["source_time"].isna()
destination_time_missing = edge_times["destination_time"].isna()
known_times = ~source_time_missing & ~destination_time_missing

print("Edges with missing source time:", int(source_time_missing.sum()))
print("Edges with missing destination time:",
      int(destination_time_missing.sum()))
print("Edges with both endpoint times missing:",
      int((source_time_missing & destination_time_missing).sum()))
print("Edges with either endpoint time missing:",
      int((source_time_missing | destination_time_missing).sum()))

relation = pd.Series("missing endpoint time", index=edge_times.index)
relation.loc[
    known_times & (edge_times["source_time"] < edge_times["destination_time"])
] = "source earlier step"
relation.loc[
    known_times & (edge_times["source_time"] == edge_times["destination_time"])
] = "same step"
relation.loc[
    known_times & (edge_times["source_time"] > edge_times["destination_time"])
] = "source later step"

print("\nEdge time-step relationships:")
display(relation.value_counts().rename("unique_edge_count").to_frame())

# Count unique incoming and outgoing neighbors from the full supplied graph.
full_in_degree = unique_edges.groupby("txId2")["txId1"].nunique()
full_out_degree = unique_edges.groupby("txId1")["txId2"].nunique()

train_nodes = train[["txId", "time_step"]].copy()
train_nodes["split"] = "train"
test_nodes = test[["txId", "time_step"]].copy()
test_nodes["split"] = "test"

node_table = (
    pd.concat([train_nodes, test_nodes], ignore_index=True)
    .drop_duplicates("txId")
)
node_table["in_degree"] = node_table["txId"].map(full_in_degree).fillna(0)
node_table["out_degree"] = node_table["txId"].map(full_out_degree).fillna(0)

display(
    node_table.groupby("split")[["in_degree", "out_degree"]]
    .describe(percentiles=[0.5, 0.9, 0.99])
)

# For labeled train transactions, count how many direct neighbors have features
# available in train.csv or test.csv.
feature_ids = set(node_table["txId"])
labeled_ids = set(labeled["txId"])

incoming_total = (
    unique_edges.loc[unique_edges["txId2"].isin(labeled_ids)]
    .groupby("txId2")["txId1"]
    .nunique()
)
incoming_with_features = (
    unique_edges.loc[
        unique_edges["txId2"].isin(labeled_ids)
        & unique_edges["txId1"].isin(feature_ids)
    ]
    .groupby("txId2")["txId1"]
    .nunique()
)
outgoing_total = (
    unique_edges.loc[unique_edges["txId1"].isin(labeled_ids)]
    .groupby("txId1")["txId2"]
    .nunique()
)
outgoing_with_features = (
    unique_edges.loc[
        unique_edges["txId1"].isin(labeled_ids)
        & unique_edges["txId2"].isin(feature_ids)
    ]
    .groupby("txId1")["txId2"]
    .nunique()
)

neighbor_coverage = pd.DataFrame(index=pd.Index(labeled["txId"], name="txId"))
neighbor_coverage["incoming_neighbors"] = (
    incoming_total.reindex(neighbor_coverage.index).fillna(0).to_numpy()
)
neighbor_coverage["incoming_neighbors_with_features"] = (
    incoming_with_features.reindex(neighbor_coverage.index).fillna(0).to_numpy()
)
neighbor_coverage["outgoing_neighbors"] = (
    outgoing_total.reindex(neighbor_coverage.index).fillna(0).to_numpy()
)
neighbor_coverage["outgoing_neighbors_with_features"] = (
    outgoing_with_features.reindex(neighbor_coverage.index).fillna(0).to_numpy()
)

coverage_summary = pd.DataFrame(
    {
        "labeled_transactions": [len(neighbor_coverage)],
        "with_incoming_feature_neighbor": [
            (neighbor_coverage["incoming_neighbors_with_features"] > 0).sum()
        ],
        "share_with_incoming_feature_neighbor": [
            (neighbor_coverage["incoming_neighbors_with_features"] > 0).mean()
        ],
        "with_outgoing_feature_neighbor": [
            (neighbor_coverage["outgoing_neighbors_with_features"] > 0).sum()
        ],
        "share_with_outgoing_feature_neighbor": [
            (neighbor_coverage["outgoing_neighbors_with_features"] > 0).mean()
        ],
    }
)
display(coverage_summary)

Raw edge rows: 234355
Edge rows with missing endpoint IDs: 0
Duplicate directed edge rows: 0
Unique directed edges: 234355
Self-loops: 0
Edges with missing source time: 52398
Edges with missing destination time: 40231
Edges with both endpoint times missing: 34192
Edges with either endpoint time missing: 58437

Edge time-step relationships:


,unique_edge_count
same step,175918
missing endpoint time,58437


in_degree                                                 out_degree  \
          count      mean       std  min  50%  90%   99%    max      count   
split                                                                        
test    15329.0  2.017744  8.646258  0.0  1.0  2.0  42.0  241.0    15329.0   
train  141772.0  1.151102  3.568221  0.0  1.0  2.0   9.0  284.0   141772.0   

                                                     
           mean      std  min  50%  90%  99%    max  
split                                                
test   1.224020  1.78621  0.0  1.0  2.0  9.0   99.0  
train  1.151102  1.96091  0.0  1.0  2.0  5.0  472.0

,labeled_transactions,with_incoming_feature_neighbor,share_with_incoming_feature_neighbor,with_outgoing_feature_neighbor,share_with_outgoing_feature_neighbor
0,31235,21532,0.689355,22915,0.733632


In [10]:
# step 5: Compare feature distributions across time

print("Candidate canonical grouping, if original column order is preserved:")
print("Local candidates: feat_1 through feat_93")
print("Aggregate candidates: feat_94 through feat_165")
print("Counts:", 93, "and", 72)

period_frames = {
    "train_steps_1_21": train.loc[train["time_step"].between(1, 21)],
    "train_steps_22_28": train.loc[train["time_step"].between(22, 28)],
    "train_steps_29_35": train.loc[train["time_step"].between(29, 35)],
    "test_steps_36_49": test,
}

summary_rows = []
for period_name, frame in period_frames.items():
    for feature in numeric_features:
        values = frame[feature]
        observed = values.dropna()

        summary_rows.append(
            {
                "period": period_name,
                "feature": feature,
                "row_count": len(frame),
                "missing_fraction": values.isna().mean(),
                "mean": observed.mean(),
                "std": observed.std(),
                "median": observed.median(),
                "zero_fraction": observed.eq(0).mean(),
                "q01": observed.quantile(0.01),
                "q99": observed.quantile(0.99),
            }
        )

feature_period_summary = pd.DataFrame(summary_rows)
display(feature_period_summary.head(30))

# Rank features whose test-period means differ most from the latest labeled era.
means = feature_period_summary.pivot(
    index="feature", columns="period", values="mean"
)
stds = feature_period_summary.pivot(
    index="feature", columns="period", values="std"
)
zero_rates = feature_period_summary.pivot(
    index="feature", columns="period", values="zero_fraction"
)

drift = pd.DataFrame(index=means.index)
drift["train_late_mean"] = means["train_steps_29_35"]
drift["test_mean"] = means["test_steps_36_49"]
drift["train_late_std"] = stds["train_steps_29_35"]
drift["standardized_mean_shift"] = (
    (drift["test_mean"] - drift["train_late_mean"])
    / drift["train_late_std"].replace(0, np.nan)
)
drift["absolute_standardized_mean_shift"] = (
    drift["standardized_mean_shift"].abs()
)
drift["train_late_zero_fraction"] = zero_rates["train_steps_29_35"]
drift["test_zero_fraction"] = zero_rates["test_steps_36_49"]
drift["zero_fraction_change"] = (
    drift["test_zero_fraction"] - drift["train_late_zero_fraction"]
)

display(
    drift.sort_values(
        "absolute_standardized_mean_shift", ascending=False
    ).head(20)
)

Candidate canonical grouping, if original column order is preserved:
Local candidates: feat_1 through feat_93
Aggregate candidates: feat_94 through feat_165
Counts: 93 and 72


,period,feature,row_count,missing_fraction,mean,std,median,zero_fraction,q01,q99
0,train_steps_1_21,feat_1,94299,0.0,0.006334,0.655792,-0.168054,0.0,-0.172976,2.546358
1,train_steps_1_21,feat_2,94299,0.0,-0.140844,0.217436,-0.178377,0.0,-0.198853,0.215308
2,train_steps_1_21,feat_3,94299,0.0,-0.533546,0.909904,-1.201369,0.0,-1.201369,1.573595
3,train_steps_1_21,feat_4,94299,0.0,-0.000286,0.954532,-0.121970,0.0,-0.121970,1.678931
4,train_steps_1_21,feat_5,94299,0.0,-0.000100,1.169582,-0.043875,0.0,-0.063725,0.333276
5,train_steps_1_21,feat_6,94299,0.0,-0.005253,0.935907,-0.113002,0.0,-0.113002,1.482058
6,train_steps_1_21,feat_7,94299,0.0,0.015717,1.007408,-0.061584,0.0,-0.061584,1.459897
7,train_steps_1_21,feat_8,94299,0.0,0.006549,0.650216,-0.161619,0.0,-0.163645,2.590610
8,train_steps_1_21,feat_9,94299,0.0,0.005846,0.651114,-0.165153,0.0,-0.169454,2.556344
9,train_steps_1_21,feat_10,94299,0.0,-0.002842,0.662184,-0.049707,0.0,-0.049707,0.728709


,train_late_mean,test_mean,train_late_std,standardized_mean_shift,absolute_standardized_mean_shift,train_late_zero_fraction,test_zero_fraction,zero_fraction_change
feature,,,,,,,,
feat_101,0.545352,1.185889,0.178000,3.598523,3.598523,0.0,0.0,0.0
feat_103,0.543815,1.184195,0.190420,3.362984,3.362984,0.0,0.0,0.0
feat_100,0.537916,1.172057,0.232518,2.727273,2.727273,0.0,0.0,0.0
feat_109,-0.061867,1.004629,0.465065,2.293217,2.293217,0.0,0.0,0.0
feat_107,-0.069758,1.123706,0.543863,2.194420,2.194420,0.0,0.0,0.0
feat_106,-0.053039,0.886668,0.432057,2.174961,2.174961,0.0,0.0,0.0
feat_108,-0.046826,0.613110,0.668591,0.987055,0.987055,0.0,0.0,0.0
feat_113,0.278686,0.968496,0.787358,0.876107,0.876107,0.0,0.0,0.0
feat_136,0.480845,1.089906,0.702092,0.867495,0.867495,0.0,0.0,0.0


In [11]:
# step 6: Test label homophily with a time-step-stratified shuffle

# Keep only train transactions with known labels.
labeled_nodes = train.loc[
    label_numeric.notna(), ["txId", "time_step"]
].copy()
labeled_nodes["label"] = label_numeric.loc[labeled_nodes.index].astype(int)

label_position = pd.Series(
    np.arange(len(labeled_nodes)),
    index=labeled_nodes["txId"],
)

# Keep edges whose two endpoint labels are known.
eligible_edges = unique_edges.loc[
    (unique_edges["txId1"] != unique_edges["txId2"])
    & unique_edges["txId1"].isin(label_position.index)
    & unique_edges["txId2"].isin(label_position.index)
].copy()

if eligible_edges.empty:
    print("No edges have labeled transactions at both endpoints.")
else:
    source_pos = eligible_edges["txId1"].map(label_position).to_numpy(dtype=int)
    destination_pos = eligible_edges["txId2"].map(label_position).to_numpy(dtype=int)
    labels = labeled_nodes["label"].to_numpy()

    observed_same_label = labels[source_pos] == labels[destination_pos]
    observed_rate = observed_same_label.mean()

    # Shuffle labels only among labeled transactions in the same time step.
    # This preserves time-step class prevalence while retaining the graph.
    time_values = labeled_nodes["time_step"].to_numpy()
    positions_by_time = [
        np.flatnonzero(time_values == step)
        for step in np.unique(time_values)
    ]

    rng = np.random.default_rng(42)
    n_permutations = 1000
    shuffled_rates = np.empty(n_permutations)

    for permutation in range(n_permutations):
        shuffled_labels = labels.copy()

        for positions in positions_by_time:
            shuffled_labels[positions] = rng.permutation(labels[positions])

        shuffled_rates[permutation] = (
            shuffled_labels[source_pos] == shuffled_labels[destination_pos]
        ).mean()

    difference = observed_rate - shuffled_rates.mean()
    p_value = (
        1 + np.count_nonzero(shuffled_rates >= observed_rate)
    ) / (n_permutations + 1)

    print("Unique edges with labels at both endpoints:", len(eligible_edges))
    print(f"Observed same-label rate: {observed_rate:.4f}")
    print(f"Within-time-step shuffle mean: {shuffled_rates.mean():.4f}")
    print(f"Observed minus shuffle mean: {difference:+.4f}")
    print(
        "Shuffle 2.5%, 50%, 97.5% quantiles:",
        np.quantile(shuffled_rates, [0.025, 0.50, 0.975]).round(4),
    )
    print(f"One-sided permutation p-value: {p_value:.4f}")

# strong evidence of label homophily among the 23,900 edges whose endpoints both have known labels

Unique edges with labels at both endpoints: 23900
Observed same-label rate: 0.9468
Within-time-step shuffle mean: 0.8188
Observed minus shuffle mean: +0.1281
Shuffle 2.5%, 50%, 97.5% quantiles: [0.803  0.8199 0.8269]
One-sided permutation p-value: 0.0010


In [12]:
# Use only rows with known labels.
baseline_data = train.loc[train["label"].notna()].copy()
baseline_data["label"] = pd.to_numeric(baseline_data["label"]).astype(int)

candidate_local = [f"feat_{i}" for i in range(1, 94)]
candidate_aggregate = [f"feat_{i}" for i in range(94, 166)]
all_features = [f"feat_{i}" for i in range(1, 166)]

feature_sets = {
    "candidate_local_93": candidate_local,
    "candidate_aggregate_72": candidate_aggregate,
    "all_165": all_features,
}

# Expanding chronological validation folds.
temporal_folds = [
    ("steps 22–28", 21, 22, 28),
    ("steps 29–35", 28, 29, 35),
]

results = []

for fold_name, last_train_step, first_valid_step, last_valid_step in temporal_folds:
    fold_train = baseline_data.loc[
        baseline_data["time_step"] <= last_train_step
    ]
    fold_valid = baseline_data.loc[
        baseline_data["time_step"].between(first_valid_step, last_valid_step)
    ]

    print(
        f"{fold_name}: train rows={len(fold_train):,}, "
        f"validation rows={len(fold_valid):,}, "
        f"validation illicit={int(fold_valid['label'].sum()):,}"
    )

    for feature_set_name, columns in feature_sets.items():
        model = HistGradientBoostingClassifier(
            max_iter=200,
            learning_rate=0.08,
            max_leaf_nodes=31,
            early_stopping=False,
            random_state=42,
        )
        model.fit(fold_train[columns], fold_train["label"])
        probabilities = model.predict_proba(fold_valid[columns])[:, 1]

        results.append(
            {
                "fold": fold_name,
                "feature_set": feature_set_name,
                "roc_auc": roc_auc_score(
                    fold_valid["label"], probabilities
                ),
            }
        )

baseline_results = pd.DataFrame(results)

display(
    baseline_results.pivot(
        index="feature_set",
        columns="fold",
        values="roc_auc",
    )
)

display(
    baseline_results.groupby("feature_set")["roc_auc"]
    .mean()
    .sort_values(ascending=False)
    .rename("mean_temporal_auc")
    .to_frame()
)

steps 22–28: train rows=19,530, validation rows=5,677, validation illicit=671
steps 29–35: train rows=25,207, validation rows=6,028, validation illicit=1,102


fold,steps 22–28,steps 29–35
feature_set,,
all_165,0.942545,0.996665
candidate_aggregate_72,0.938903,0.949814
candidate_local_93,0.943828,0.993067


,mean_temporal_auc
feature_set,
all_165,0.969605
candidate_local_93,0.968448
candidate_aggregate_72,0.944359


In [13]:
# Compare local features with features summarized from direct predecessors.
# Assumes txId1 -> txId2 means an output from txId1 is spent by txId2.

LOCAL_FEATURES = [f"feat_{i}" for i in range(1, 94)]
ALL_FEATURES = [f"feat_{i}" for i in range(1, 166)]

# One feature row per transaction, including unlabeled train and test transactions.
node_features = pd.concat(
    [
        train[["txId"] + LOCAL_FEATURES],
        test[["txId"] + LOCAL_FEATURES],
    ],
    ignore_index=True,
).drop_duplicates("txId")

# Use unique directed edges only.
unique_edges = (
    edges[["txId1", "txId2"]]
    .dropna(subset=["txId1", "txId2"])
    .drop_duplicates()
)

# Total incoming neighbors, including those without a feature row in train/test.
incoming_total = unique_edges.groupby("txId2")["txId1"].nunique()

# Keep edges whose predecessor has a feature row, then summarize predecessor
# local features for each destination transaction.
parent_edges = unique_edges.merge(
    node_features,
    left_on="txId1",
    right_on="txId",
    how="inner",
    validate="many_to_one",
).drop(columns="txId")

incoming_available = parent_edges.groupby("txId2")["txId1"].nunique()
incoming_means = parent_edges.groupby("txId2")[LOCAL_FEATURES].mean()
incoming_means.columns = [
    f"in_parent_mean_{c}" for c in LOCAL_FEATURES
]
incoming_means.index.name = "txId"

# Create graph features for train and test transaction IDs.
all_ids = pd.Index(
    pd.concat([train["txId"], test["txId"]], ignore_index=True).drop_duplicates(),
    name="txId",
)
graph_features = pd.DataFrame(index=all_ids)

graph_features["incoming_degree"] = (
    graph_features.index.to_series()
    .map(incoming_total)
    .fillna(0)
    .to_numpy()
)
graph_features["log_incoming_degree"] = np.log1p(
    graph_features["incoming_degree"]
)
graph_features["incoming_feature_neighbor_count"] = (
    graph_features.index.to_series()
    .map(incoming_available)
    .fillna(0)
    .to_numpy()
)
graph_features["incoming_feature_coverage"] = np.divide(
    graph_features["incoming_feature_neighbor_count"],
    graph_features["incoming_degree"],
    out=np.zeros(len(graph_features), dtype=float),
    where=graph_features["incoming_degree"].to_numpy() > 0,
)

graph_features = graph_features.join(incoming_means, how="left")
incoming_mean_cols = incoming_means.columns.tolist()
graph_features[incoming_mean_cols] = graph_features[incoming_mean_cols].fillna(0)

GRAPH_FEATURES = graph_features.columns.tolist()

print("Transactions with graph features:", len(graph_features))
print("Unique incoming edges with predecessor features:", len(parent_edges))
print(
    "Transactions with at least one incoming neighbor feature row:",
    int((graph_features["incoming_feature_neighbor_count"] > 0).sum()),
)

# Join graph features to the labeled training rows by txId.
labeled_graph = train.loc[train["label"].notna()].copy()
labeled_graph["label"] = pd.to_numeric(labeled_graph["label"]).astype(int)
labeled_graph = labeled_graph.merge(
    graph_features.reset_index(),
    on="txId",
    how="left",
    validate="one_to_one",
    sort=False,
)

feature_sets = {
    "candidate_local_93": LOCAL_FEATURES,
    "local_plus_incoming_graph": LOCAL_FEATURES + GRAPH_FEATURES,
    "all_165_diagnostic": ALL_FEATURES,
}

temporal_folds = [
    ("steps 22–28", 21, 22, 28),
    ("steps 29–35", 28, 29, 35),
]

rows = []
for fold_name, last_train_step, first_valid_step, last_valid_step in temporal_folds:
    fold_train = labeled_graph.loc[
        labeled_graph["time_step"] <= last_train_step
    ]
    fold_valid = labeled_graph.loc[
        labeled_graph["time_step"].between(first_valid_step, last_valid_step)
    ]

    for feature_set_name, columns in feature_sets.items():
        model = HistGradientBoostingClassifier(
            max_iter=200,
            learning_rate=0.08,
            max_leaf_nodes=31,
            early_stopping=False,
            random_state=42,
        )
        model.fit(fold_train[columns], fold_train["label"])
        probabilities = model.predict_proba(fold_valid[columns])[:, 1]

        rows.append(
            {
                "fold": fold_name,
                "feature_set": feature_set_name,
                "roc_auc": roc_auc_score(
                    fold_valid["label"], probabilities
                ),
            }
        )

graph_comparison = pd.DataFrame(rows)
display(
    graph_comparison.pivot(
        index="feature_set",
        columns="fold",
        values="roc_auc",
    )
)
# The incoming-neighbor features improved AUC on both folds: +0.0016 on steps 22–28 and +0.0024 on steps 29–35 versus local features alone

Transactions with graph features: 157101
Unique incoming edges with predecessor features: 181957
Transactions with at least one incoming neighbor feature row: 111613


fold,steps 22–28,steps 29–35
feature_set,,
all_165_diagnostic,0.942545,0.996665
candidate_local_93,0.943828,0.993067
local_plus_incoming_graph,0.945393,0.995516


In [14]:
parent_mean_features = [
    c for c in GRAPH_FEATURES
    if c.startswith("in_parent_mean_")
]

degree_features = [
    "incoming_degree",
    "log_incoming_degree",
]

coverage_features = [
    "incoming_feature_neighbor_count",
    "incoming_feature_coverage",
]

feature_sets = {
    "local_only": LOCAL_FEATURES,
    "local_plus_degree": LOCAL_FEATURES + degree_features,
    "local_plus_coverage": LOCAL_FEATURES + coverage_features,
    "local_plus_parent_means": LOCAL_FEATURES + parent_mean_features,
    "local_plus_all_incoming": LOCAL_FEATURES + GRAPH_FEATURES,
    "all_165_diagnostic": ALL_FEATURES,
}

rows = []

for fold_name, last_train_step, first_valid_step, last_valid_step in temporal_folds:
    fold_train = labeled_graph.loc[
        labeled_graph["time_step"] <= last_train_step
    ]
    fold_valid = labeled_graph.loc[
        labeled_graph["time_step"].between(first_valid_step, last_valid_step)
    ]

    for feature_set_name, columns in feature_sets.items():
        model = HistGradientBoostingClassifier(
            max_iter=200,
            learning_rate=0.08,
            max_leaf_nodes=31,
            early_stopping=False,
            random_state=42,
        )
        model.fit(fold_train[columns], fold_train["label"])
        probabilities = model.predict_proba(fold_valid[columns])[:, 1]

        rows.append(
            {
                "fold": fold_name,
                "feature_set": feature_set_name,
                "roc_auc": roc_auc_score(
                    fold_valid["label"], probabilities
                ),
            }
        )

ablation = pd.DataFrame(rows)
auc_by_fold = ablation.pivot(
    index="feature_set",
    columns="fold",
    values="roc_auc",
)

fold_names = [fold[0] for fold in temporal_folds]
auc_by_fold["mean_auc"] = auc_by_fold[fold_names].mean(axis=1)

for fold_name in fold_names:
    auc_by_fold[f"gain_vs_local_{fold_name}"] = (
        auc_by_fold[fold_name] - auc_by_fold.loc["local_only", fold_name]
    )

display(auc_by_fold.sort_values("mean_auc", ascending=False))

# The incoming graph features give a small lift on both folds. 
# Counts/coverage help more on steps 22–28, while predecessor feature means help more on steps 29–35.

fold,steps 22–28,steps 29–35,mean_auc,gain_vs_local_steps 22–28,gain_vs_local_steps 29–35
feature_set,,,,,
local_plus_coverage,0.948329,0.993375,0.970852,0.004501,0.000308
local_plus_degree,0.948329,0.993375,0.970852,0.004501,0.000308
local_plus_all_incoming,0.945393,0.995516,0.970455,0.001565,0.002449
local_plus_parent_means,0.944709,0.995303,0.970006,0.000881,0.002236
all_165_diagnostic,0.942545,0.996665,0.969605,-0.001283,0.003598
local_only,0.943828,0.993067,0.968448,0.000000,0.000000


In [15]:
# Summarize the spread of predecessor local features for each destination.
# parent_edges and graph_features were created in the previous cell.

extra_stat_tables = []

for stat_name, aggregation in [
    ("std", "std"),
    ("min", "min"),
    ("max", "max"),
]:
    stat_table = (
        parent_edges.groupby("txId2")[LOCAL_FEATURES]
        .agg(aggregation)
    )
    stat_table.columns = [
        f"in_parent_{stat_name}_{c}" for c in LOCAL_FEATURES
    ]
    stat_table.index.name = "txId"
    extra_stat_tables.append(stat_table)

parent_extra = pd.concat(extra_stat_tables, axis=1)
parent_extra = parent_extra.reindex(graph_features.index).fillna(0)
extra_stat_features = parent_extra.columns.tolist()

# Add these summaries to the labeled rows by transaction ID.
labeled_extended = labeled_graph.merge(
    parent_extra.reset_index(),
    on="txId",
    how="left",
    validate="one_to_one",
    sort=False,
)
labeled_extended[extra_stat_features] = (
    labeled_extended[extra_stat_features].fillna(0)
)

parent_mean_features = [
    c for c in GRAPH_FEATURES
    if c.startswith("in_parent_mean_")
]
count_coverage_features = [
    "incoming_degree",
    "log_incoming_degree",
    "incoming_feature_neighbor_count",
    "incoming_feature_coverage",
]

feature_sets = {
    "local_only": LOCAL_FEATURES,
    "local_plus_counts_coverage": LOCAL_FEATURES + count_coverage_features,
    "local_plus_parent_means": LOCAL_FEATURES + parent_mean_features,
    "local_plus_all_incoming_means": LOCAL_FEATURES + GRAPH_FEATURES,
    "local_plus_incoming_mean_and_spread": (
        LOCAL_FEATURES + GRAPH_FEATURES + extra_stat_features
    ),
    "all_165_diagnostic": ALL_FEATURES,
}

temporal_folds = [
    ("steps 22–28", 21, 22, 28),
    ("steps 29–35", 28, 29, 35),
]

rows = []

for fold_name, last_train_step, first_valid_step, last_valid_step in temporal_folds:
    fold_train = labeled_extended.loc[
        labeled_extended["time_step"] <= last_train_step
    ]
    fold_valid = labeled_extended.loc[
        labeled_extended["time_step"].between(
            first_valid_step, last_valid_step
        )
    ]

    for feature_set_name, columns in feature_sets.items():
        model = HistGradientBoostingClassifier(
            max_iter=200,
            learning_rate=0.08,
            max_leaf_nodes=31,
            early_stopping=False,
            random_state=42,
        )
        model.fit(fold_train[columns], fold_train["label"])
        probabilities = model.predict_proba(fold_valid[columns])[:, 1]

        rows.append(
            {
                "fold": fold_name,
                "feature_set": feature_set_name,
                "roc_auc": roc_auc_score(
                    fold_valid["label"], probabilities
                ),
            }
        )

incoming_stat_results = pd.DataFrame(rows)
auc_by_fold = incoming_stat_results.pivot(
    index="feature_set",
    columns="fold",
    values="roc_auc",
)

fold_names = [fold[0] for fold in temporal_folds]
auc_by_fold["mean_auc"] = auc_by_fold[fold_names].mean(axis=1)

display(auc_by_fold.sort_values("mean_auc", ascending=False))

fold,steps 22–28,steps 29–35,mean_auc
feature_set,,,
local_plus_incoming_mean_and_spread,0.946848,0.995208,0.971028
local_plus_counts_coverage,0.948329,0.993375,0.970852
local_plus_all_incoming_means,0.945393,0.995516,0.970455
local_plus_parent_means,0.944709,0.995303,0.970006
all_165_diagnostic,0.942545,0.996665,0.969605
local_only,0.943828,0.993067,0.968448


In [16]:
selected_features = (
    LOCAL_FEATURES
    + GRAPH_FEATURES
    + extra_stat_features
)

weighting_rows = []

for fold_name, last_train_step, first_valid_step, last_valid_step in temporal_folds:
    fold_train = labeled_extended.loc[
        labeled_extended["time_step"] <= last_train_step
    ]
    fold_valid = labeled_extended.loc[
        labeled_extended["time_step"].between(
            first_valid_step, last_valid_step
        )
    ]

    y_train = fold_train["label"].to_numpy()
    n_negative = int((y_train == 0).sum())
    n_positive = int((y_train == 1).sum())

    balanced_weights = np.where(
        y_train == 1,
        len(y_train) / (2 * n_positive),
        len(y_train) / (2 * n_negative),
    )

    for weighting_name, sample_weight in [
        ("unweighted", None),
        ("balanced_sample_weight", balanced_weights),
    ]:
        model = HistGradientBoostingClassifier(
            max_iter=200,
            learning_rate=0.08,
            max_leaf_nodes=31,
            early_stopping=False,
            random_state=42,
        )
        model.fit(
            fold_train[selected_features],
            y_train,
            sample_weight=sample_weight,
        )
        probabilities = model.predict_proba(
            fold_valid[selected_features]
        )[:, 1]

        weighting_rows.append(
            {
                "fold": fold_name,
                "weighting": weighting_name,
                "roc_auc": roc_auc_score(
                    fold_valid["label"], probabilities
                ),
            }
        )

weighting_results = pd.DataFrame(weighting_rows)
weighting_table = weighting_results.pivot(
    index="weighting",
    columns="fold",
    values="roc_auc",
)
weighting_table["mean_auc"] = weighting_table.mean(axis=1)

display(weighting_table)

fold,steps 22–28,steps 29–35,mean_auc
weighting,,,
balanced_sample_weight,0.948266,0.995450,0.971858
unweighted,0.946848,0.995208,0.971028


# model exploration

In [20]:
selected_features = (
    LOCAL_FEATURES
    + GRAPH_FEATURES
    + extra_stat_features
)

def make_models():
    models = {
        "Logistic Regression": Pipeline(
            [
                ("scale", StandardScaler()),
                ("model", LogisticRegression(max_iter=3000, random_state=42)),
            ]
        ),
        "Decision Tree": DecisionTreeClassifier(
            max_depth=8,
            min_samples_leaf=20,
            random_state=42,
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            max_features="sqrt",
            n_jobs=-1,
            random_state=42,
        ),
        "Extra Trees": ExtraTreesClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            max_features="sqrt",
            n_jobs=-1,
            random_state=42,
        ),
        "Histogram Gradient Boosting": HistGradientBoostingClassifier(
            max_iter=200,
            learning_rate=0.08,
            max_leaf_nodes=31,
            early_stopping=False,
            random_state=42,
        ),
    }

    if LGBMClassifier is not None:
        models["LightGBM"] = LGBMClassifier(
            n_estimators=600,
            learning_rate=0.03,
            num_leaves=31,
            colsample_bytree=0.85,
            subsample=0.85,
            subsample_freq=1,
            random_state=42,
            n_jobs=-1,
            verbose=-1,
        )

    return models


model_rows = []

for fold_name, last_train_step, first_valid_step, last_valid_step in temporal_folds:
    fold_train = labeled_extended.loc[
        labeled_extended["time_step"] <= last_train_step
    ]
    fold_valid = labeled_extended.loc[
        labeled_extended["time_step"].between(
            first_valid_step, last_valid_step
        )
    ]

    y_train = fold_train["label"].to_numpy()
    n_negative = int((y_train == 0).sum())
    n_positive = int((y_train == 1).sum())
    balanced_weights = np.where(
        y_train == 1,
        len(y_train) / (2 * n_positive),
        len(y_train) / (2 * n_negative),
    )

    for model_name, model in make_models().items():
        if model_name == "Logistic Regression":
            model.fit(
                fold_train[selected_features],
                y_train,
                model__sample_weight=balanced_weights,
            )
        else:
            model.fit(
                fold_train[selected_features],
                y_train,
                sample_weight=balanced_weights,
            )

        probabilities = model.predict_proba(
            fold_valid[selected_features]
        )[:, 1]

        model_rows.append(
            {
                "fold": fold_name,
                "model": model_name,
                "roc_auc": roc_auc_score(
                    fold_valid["label"], probabilities
                ),
            }
        )

model_results = pd.DataFrame(model_rows)
model_auc = model_results.pivot(
    index="model",
    columns="fold",
    values="roc_auc",
)
model_auc["mean_auc"] = model_auc.mean(axis=1)

display(model_auc.sort_values("mean_auc", ascending=False))

fold,steps 22–28,steps 29–35,mean_auc
model,,,
Histogram Gradient Boosting,0.948266,0.995450,0.971858
LightGBM,0.935622,0.996196,0.965909
Extra Trees,0.915247,0.992648,0.953947
Random Forest,0.896606,0.995372,0.945989
Logistic Regression,0.869400,0.914313,0.891857
Decision Tree,0.874758,0.775454,0.825106


In [23]:
# GNN
import torch
from torch import nn
from torch.nn import functional as F
from sklearn.preprocessing import StandardScaler

# Candidate local features only; the 93/72 feature split remains an assumption.
LOCAL_FEATURES = [f"feat_{i}" for i in range(1, 94)]

# Create one feature row for every transaction available in train or test.
node_features_df = pd.concat(
    [
        train[["txId"] + LOCAL_FEATURES],
        test[["txId"] + LOCAL_FEATURES],
    ],
    ignore_index=True,
).drop_duplicates("txId").reset_index(drop=True)

node_to_position = pd.Series(
    np.arange(len(node_features_df)),
    index=node_features_df["txId"],
)

# Keep directed edges only when both endpoint feature rows are available.
edge_pairs = (
    edges[["txId1", "txId2"]]
    .dropna(subset=["txId1", "txId2"])
    .drop_duplicates()
)
source_positions = edge_pairs["txId1"].map(node_to_position)
destination_positions = edge_pairs["txId2"].map(node_to_position)
edge_known_mask = source_positions.notna() & destination_positions.notna()

source_positions = source_positions.loc[edge_known_mask].to_numpy(dtype=np.int64)
destination_positions = destination_positions.loc[edge_known_mask].to_numpy(dtype=np.int64)

# Labeled rows and their corresponding graph-node positions.
label_values = pd.to_numeric(train["label"], errors="coerce")
known_rows = train.loc[
    label_values.notna(), ["txId", "time_step"]
].copy()
known_rows["label"] = label_values.loc[known_rows.index].astype(int).to_numpy()
known_rows["node_position"] = (
    known_rows["txId"].map(node_to_position).astype(int).to_numpy()
)

raw_node_features = node_features_df[LOCAL_FEATURES].to_numpy(dtype=np.float32)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print("Nodes with feature rows:", len(node_features_df))
print("Directed edges with both endpoint feature rows:", len(source_positions))


class IncomingGraphSAGE(nn.Module):
    """Two-layer mean-aggregation model using incoming directed edges."""

    def __init__(self, input_size, hidden_size=64, dropout=0.25):
        super().__init__()
        self.self_layer_1 = nn.Linear(input_size, hidden_size)
        self.neighbor_layer_1 = nn.Linear(input_size, hidden_size, bias=False)
        self.self_layer_2 = nn.Linear(hidden_size, hidden_size)
        self.neighbor_layer_2 = nn.Linear(hidden_size, hidden_size, bias=False)
        self.output_layer = nn.Linear(hidden_size, 1)
        self.dropout = dropout

    @staticmethod
    def mean_incoming(features, source_idx, destination_idx):
        aggregated = features.new_zeros(features.shape)
        aggregated.index_add_(0, destination_idx, features[source_idx])

        counts = features.new_zeros((features.shape[0], 1))
        counts.index_add_(
            0,
            destination_idx,
            features.new_ones((len(destination_idx), 1)),
        )
        return aggregated / counts.clamp_min(1)

    def forward(self, features, source_idx, destination_idx):
        incoming_1 = self.mean_incoming(
            features, source_idx, destination_idx
        )
        hidden = F.relu(
            self.self_layer_1(features)
            + self.neighbor_layer_1(incoming_1)
        )
        hidden = F.dropout(
            hidden, p=self.dropout, training=self.training
        )

        incoming_2 = self.mean_incoming(
            hidden, source_idx, destination_idx
        )
        hidden = F.relu(
            self.self_layer_2(hidden)
            + self.neighbor_layer_2(incoming_2)
        )
        return self.output_layer(hidden).squeeze(1)


gnn_folds = [
    ("steps 22–28", 21, 22, 28),
    ("steps 29–35", 28, 29, 35),
]

gnn_rows = []

for fold_name, last_train_step, first_valid_step, last_valid_step in gnn_folds:
    train_mask = known_rows["time_step"].to_numpy() <= last_train_step
    valid_mask = known_rows["time_step"].between(
        first_valid_step, last_valid_step
    ).to_numpy()

    fold_train = known_rows.loc[train_mask]
    fold_valid = known_rows.loc[valid_mask]

    train_node_idx = fold_train["node_position"].to_numpy(dtype=np.int64)
    valid_node_idx = fold_valid["node_position"].to_numpy(dtype=np.int64)
    y_train = fold_train["label"].to_numpy(dtype=np.float32)
    y_valid = fold_valid["label"].to_numpy(dtype=int)

    # Fit scaling on labeled training nodes in this fold only.
    scaler = StandardScaler()
    scaler.fit(raw_node_features[train_node_idx])
    scaled_features = scaler.transform(raw_node_features).astype(np.float32)

    x_tensor = torch.tensor(scaled_features, dtype=torch.float32, device=device)
    src_tensor = torch.tensor(source_positions, dtype=torch.long, device=device)
    dst_tensor = torch.tensor(destination_positions, dtype=torch.long, device=device)
    train_idx_tensor = torch.tensor(train_node_idx, dtype=torch.long, device=device)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32, device=device)

    n_negative = int((y_train == 0).sum())
    n_positive = int((y_train == 1).sum())
    positive_weight = torch.tensor(
        [n_negative / n_positive],
        dtype=torch.float32,
        device=device,
    )

    torch.manual_seed(42)
    model = IncomingGraphSAGE(
        input_size=len(LOCAL_FEATURES),
        hidden_size=64,
        dropout=0.25,
    ).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=0.001,
        weight_decay=1e-4,
    )
    loss_function = nn.BCEWithLogitsLoss(pos_weight=positive_weight)

    # Fixed training duration: the validation labels do not control stopping.
    for epoch in range(20):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        logits = model(x_tensor, src_tensor, dst_tensor)
        loss = loss_function(logits[train_idx_tensor], y_train_tensor)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

    model.eval()
    with torch.no_grad():
        all_logits = model(x_tensor, src_tensor, dst_tensor)
        valid_probabilities = torch.sigmoid(
            all_logits[
                torch.tensor(valid_node_idx, dtype=torch.long, device=device)
            ]
        ).cpu().numpy()

    auc = roc_auc_score(y_valid, valid_probabilities)
    gnn_rows.append(
        {
            "fold": fold_name,
            "model": "Incoming GraphSAGE",
            "roc_auc": auc,
        }
    )
    print(f"{fold_name} ROC-AUC: {auc:.6f}")

gnn_results = pd.DataFrame(gnn_rows)
display(gnn_results)

Using device: cpu
Nodes with feature rows: 157101
Directed edges with both endpoint feature rows: 175918
steps 22–28 ROC-AUC: 0.844747
steps 29–35 ROC-AUC: 0.738377


,fold,model,roc_auc
0,steps 22–28,Incoming GraphSAGE,0.844747
1,steps 29–35,Incoming GraphSAGE,0.738377


# regularisation experiment

In [29]:
l2_values = [0.0, 0.001, 0.003, 0.01, 0.03, 0.05]
rows = []

for fold_name, last_train_step, first_valid_step, last_valid_step in l2_folds:
    fold_train = labeled_extended.loc[
        labeled_extended["time_step"] <= last_train_step
    ]
    fold_valid = labeled_extended.loc[
        labeled_extended["time_step"].between(
            first_valid_step, last_valid_step
        )
    ]

    y_train = fold_train["label"].to_numpy(dtype=int)
    y_valid = fold_valid["label"].to_numpy(dtype=int)

    n_negative = (y_train == 0).sum()
    n_positive = (y_train == 1).sum()
    balanced_weights = np.where(
        y_train == 1,
        len(y_train) / (2 * n_positive),
        len(y_train) / (2 * n_negative),
    )

    for l2_value in l2_values:
        model = HistGradientBoostingClassifier(
            max_iter=200,
            learning_rate=0.08,
            max_leaf_nodes=31,
            l2_regularization=l2_value,
            early_stopping=False,
            random_state=42,
        )
        model.fit(
            fold_train[selected_features],
            y_train,
            sample_weight=balanced_weights,
        )

        valid_prob = model.predict_proba(
            fold_valid[selected_features]
        )[:, 1]

        rows.append({
            "fold": fold_name,
            "l2_regularization": l2_value,
            "validation_auc": roc_auc_score(y_valid, valid_prob),
        })

fine_l2_results = pd.DataFrame(rows)

fine_l2_table = fine_l2_results.pivot(
    index="l2_regularization",
    columns="fold",
    values="validation_auc",
)
fine_l2_table["mean_temporal_auc"] = fine_l2_table[
    [fold[0] for fold in l2_folds]
].mean(axis=1)

display(fine_l2_table.sort_values("mean_temporal_auc", ascending=False))

fold,steps 22–28,steps 29–35,mean_temporal_auc
l2_regularization,,,
0.001,0.951683,0.995461,0.973572
0.010,0.950129,0.995641,0.972885
0.003,0.949345,0.995269,0.972307
0.000,0.948266,0.995450,0.971858
0.030,0.938867,0.995256,0.967061
0.050,0.938589,0.995346,0.966967


# Fit HGB and LightGBM (without regularisation)

In [24]:
# Fit the selected feature set on all known labels, then predict test.
selected_features = (
    LOCAL_FEATURES
    + GRAPH_FEATURES
    + extra_stat_features
)

# Combine graph features for all train/test transaction IDs.
all_graph_features = graph_features.join(parent_extra, how="left").fillna(0)

labeled_all = labeled_extended.copy()
y_all = labeled_all["label"].to_numpy(dtype=int)

n_negative = int((y_all == 0).sum())
n_positive = int((y_all == 1).sum())
balanced_weights_all = np.where(
    y_all == 1,
    len(y_all) / (2 * n_positive),
    len(y_all) / (2 * n_negative),
)

# Add graph features to test rows and restore original test order.
test_scored = test.copy()
test_scored["_row_order"] = np.arange(len(test_scored))
test_scored = test_scored.merge(
    all_graph_features.reset_index(),
    on="txId",
    how="left",
    validate="one_to_one",
    sort=False,
)
test_scored = (
    test_scored.sort_values("_row_order")
    .reset_index(drop=True)
)

if not test_scored["index"].equals(test["index"]):
    raise ValueError("Test row order changed while adding graph features.")

missing_model_features = [
    c for c in selected_features
    if c not in labeled_all.columns or c not in test_scored.columns
]
if missing_model_features:
    raise ValueError(f"Missing model features: {missing_model_features}")

# Primary candidate: Histogram Gradient Boosting.
hgb_final = HistGradientBoostingClassifier(
    max_iter=200,
    learning_rate=0.08,
    max_leaf_nodes=31,
    early_stopping=False,
    random_state=42,
)
hgb_final.fit(
    labeled_all[selected_features],
    y_all,
    sample_weight=balanced_weights_all,
)
hgb_test_probabilities = hgb_final.predict_proba(
    test_scored[selected_features]
)[:, 1]

# Alternative candidate: LightGBM.
lgbm_final = LGBMClassifier(
    n_estimators=600,
    learning_rate=0.03,
    num_leaves=31,
    colsample_bytree=0.85,
    subsample=0.85,
    subsample_freq=1,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)
lgbm_final.fit(
    labeled_all[selected_features],
    y_all,
    sample_weight=balanced_weights_all,
)
lgbm_test_probabilities = lgbm_final.predict_proba(
    test_scored[selected_features]
)[:, 1]

# Load sample submission if it is not already in the notebook.
if "sample_submission" not in globals():
    sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

if not sample_submission["index"].equals(test_scored["index"]):
    raise ValueError("Sample-submission and test index orders do not match.")

submission_hgb = sample_submission[["index"]].copy()
submission_hgb["target"] = hgb_test_probabilities
submission_hgb.to_csv("submission_hgb.csv", index=False)

submission_lgbm = sample_submission[["index"]].copy()
submission_lgbm["target"] = lgbm_test_probabilities
submission_lgbm.to_csv("submission_lgbm.csv", index=False)

print("Saved submission_hgb.csv and submission_lgbm.csv")
display(submission_hgb.head())
display(submission_lgbm.head())

Saved submission_hgb.csv and submission_lgbm.csv


,index,target
0,0,0.000907
1,1,0.002665
2,2,0.000098
3,3,0.000360
4,4,0.000089


,index,target
0,0,0.000714
1,1,0.002035
2,2,0.000193
3,3,0.000109
4,4,0.000080


# Fit HGB (with regularisation)

In [30]:
hgb_final_l2 = HistGradientBoostingClassifier(
    max_iter=200,
    learning_rate=0.08,
    max_leaf_nodes=31,
    l2_regularization=0.001,
    early_stopping=False,
    random_state=42,
)

hgb_final_l2.fit(
    labeled_all[selected_features],
    y_all,
    sample_weight=balanced_weights_all,
)

probabilities = hgb_final_l2.predict_proba(
    test_scored[selected_features]
)[:, 1]

submission_hgb_l2_0001 = sample_submission[["index"]].copy()
submission_hgb_l2_0001["target"] = probabilities
submission_hgb_l2_0001.to_csv(
    "submission_hgb_l2_0001.csv",
    index=False,
)

print("Saved submission_hgb_l2_0001.csv")

Saved submission_hgb_l2_0001.csv


In [31]:
submission_hgb_l2_0001

,index,target
0,0,0.002406
1,1,0.001208
2,2,0.000090
3,3,0.000597
4,4,0.000057
...,...,...
15324,15324,0.000017
15325,15325,0.001456
15326,15326,0.006725
15327,15327,0.000044


In [26]:
sample_submission
# index column, target column
# index 0-15328
# target is between 0 and 1

,index,target
0,0,0.5
1,1,0.5
2,2,0.5
3,3,0.5
4,4,0.5
...,...,...
15324,15324,0.5
15325,15325,0.5
15326,15326,0.5
15327,15327,0.5
